# Preprocessing & Feature Engineering — Olist Delivery Time Prediction

Notebook này thực hiện:
1. Đọc 7 bảng CSV gốc từ `raw/` và lọc đơn hàng `delivered` hợp lệ.
2. Làm sạch tọa độ `geolocation` (theo bounding box Brazil) và tổng hợp đơn hàng ở cấp `order_id`.
3. Tạo các đặc trưng địa lý (`haversine_km`, `is_same_state`, `route`), thời gian (`hour`, `dayofweek`, `approval_delay_hours`, `prep_sla_days`), và đặc trưng sản phẩm/phí vận chuyển.
4. Chia tập dữ liệu theo thời gian **Time-based Split (70% Train / 15% Val / 15% Test)**.
5. Tính **Out-of-Fold Smoothed Target Encoding** (chỉ fit trên Train) để chống *Data Leakage* và lưu kết quả vào `processed/`.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

RAW_DIR = 'raw'
OUT_DIR = 'processed'
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Các hàm tiện ích (Haversine & Smoothed Target Encoding)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in kilometers between two lat/lon points."""
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def smoothed_target_encode(train_series, train_target, apply_series, smoothing=10.0):
    """Apply Bayesian-smoothed target encoding fitted on train_series/train_target."""
    global_mean = float(train_target.mean())
    stats = train_target.groupby(train_series).agg(['count', 'mean'])
    smooth = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return apply_series.map(smooth).fillna(global_mean)


def add_oof_target_encoding(train_df, val_df, test_df, col, target_col, smoothing=10.0, n_splits=5):
    """Compute out-of-fold smoothed target encoding on train, and full-train encoding on val/test."""
    feat_name = f'{col}_te'
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_encoded = pd.Series(index=train_df.index, dtype=float)

    for tr_idx, oof_idx in kf.split(train_df):
        tr_part = train_df.iloc[tr_idx]
        oof_part = train_df.iloc[oof_idx]
        train_encoded.iloc[oof_idx] = smoothed_target_encode(
            tr_part[col], tr_part[target_col], oof_part[col], smoothing=smoothing
        )

    train_df[feat_name] = train_encoded
    val_df[feat_name] = smoothed_target_encode(
        train_df[col], train_df[target_col], val_df[col], smoothing=smoothing
    )
    test_df[feat_name] = smoothed_target_encode(
        train_df[col], train_df[target_col], test_df[col], smoothing=smoothing
    )
    return feat_name

## 2. Đọc & Tổng hợp dữ liệu (Data Loading & Merging)

In [ ]:
orders = pd.read_csv(os.path.join(RAW_DIR, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(RAW_DIR, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(RAW_DIR, 'olist_customers_dataset.csv'))
sellers = pd.read_csv(os.path.join(RAW_DIR, 'olist_sellers_dataset.csv'))
products = pd.read_csv(os.path.join(RAW_DIR, 'olist_products_dataset.csv'))
geolocation = pd.read_csv(os.path.join(RAW_DIR, 'olist_geolocation_dataset.csv'))
payments = pd.read_csv(os.path.join(RAW_DIR, 'olist_order_payments_dataset.csv'))

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Chỉ giữ các đơn hàng delivered hợp lệ
df = orders[orders['order_status'] == 'delivered'].copy()
df = df.dropna(subset=['order_delivered_customer_date', 'order_purchase_timestamp']).copy()

df['delivery_time_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400.0
df = df[df['delivery_time_days'] > 0].copy()

df['olist_estimated_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

# Làm sạch geolocation theo lãnh thổ Brazil
geo_valid = geolocation[
    geolocation['geolocation_lat'].between(-35.0, 6.0)
    & geolocation['geolocation_lng'].between(-75.0, -30.0)
]
geo_zip = (
    geo_valid.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']]
    .median()
    .reset_index()
)
geo_state = (
    geo_valid.groupby('geolocation_state')[['geolocation_lat', 'geolocation_lng']]
    .median()
    .reset_index()
)

# Gom nhóm order_items + products + sellers về cấp order_id
items = order_items.merge(products, on='product_id', how='left')
items = items.merge(sellers, on='seller_id', how='left')
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])
items['product_volume_cm3'] = (
    items['product_length_cm'] * items['product_height_cm'] * items['product_width_cm']
)

order_agg = (
    items.groupby('order_id')
    .agg(
        total_items=('order_item_id', 'count'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        num_unique_sellers=('seller_id', 'nunique'),
        num_unique_products=('product_id', 'nunique'),
        total_weight_g=('product_weight_g', 'sum'),
        max_weight_g=('product_weight_g', 'max'),
        total_volume_cm3=('product_volume_cm3', 'sum'),
        avg_photos_qty=('product_photos_qty', 'mean'),
        primary_seller_id=('seller_id', 'first'),
        primary_seller_zip=('seller_zip_code_prefix', 'first'),
        primary_seller_city=('seller_city', 'first'),
        primary_seller_state=('seller_state', 'first'),
        primary_category=('product_category_name', 'first'),
        min_shipping_limit=('shipping_limit_date', 'min'),
    )
    .reset_index()
)

pay_agg = (
    payments.groupby('order_id')
    .agg(
        primary_payment_type=('payment_type', 'first'),
        max_installments=('payment_installments', 'max'),
        total_payment_value=('payment_value', 'sum'),
        num_payment_methods=('payment_sequential', 'count'),
    )
    .reset_index()
)

df = df.merge(customers, on='customer_id', how='left')
df = df.merge(order_agg, on='order_id', how='left')
df = df.merge(pay_agg, on='order_id', how='left')

# Join tọa độ zip-level + fallback theo state
df = df.merge(
    geo_zip.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'
    }),
    on='customer_zip_code_prefix', how='left'
)
df = df.merge(
    geo_zip.rename(columns={
        'geolocation_zip_code_prefix': 'primary_seller_zip',
        'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'
    }),
    on='primary_seller_zip', how='left'
)

state_lat_map = geo_state.set_index('geolocation_state')['geolocation_lat']
state_lng_map = geo_state.set_index('geolocation_state')['geolocation_lng']
df['cust_lat'] = df['cust_lat'].fillna(df['customer_state'].map(state_lat_map))
df['cust_lng'] = df['cust_lng'].fillna(df['customer_state'].map(state_lng_map))
df['sell_lat'] = df['sell_lat'].fillna(df['primary_seller_state'].map(state_lat_map))
df['sell_lng'] = df['sell_lng'].fillna(df['primary_seller_state'].map(state_lng_map))

print(f'Merged shape: {df.shape}')

## 3. Tạo đặc trưng (Feature Engineering)

In [ ]:
# Đặc trưng địa lý
df['haversine_km'] = haversine_km(df['sell_lat'], df['sell_lng'], df['cust_lat'], df['cust_lng'])
df['is_same_state'] = (df['customer_state'] == df['primary_seller_state']).astype(int)
df['is_same_city'] = (
    df['customer_city'].astype(str).str.lower() == df['primary_seller_city'].astype(str).str.lower()
).astype(int)
df['route'] = df['primary_seller_state'].fillna('UNK') + '_' + df['customer_state'].fillna('UNK')

# Đặc trưng thời gian (chỉ dùng thông tin tại thời điểm đặt/duyệt đơn)
ts = df['order_purchase_timestamp']
df['purchase_hour'] = ts.dt.hour
df['purchase_dayofweek'] = ts.dt.day_of_week
df['purchase_day'] = ts.dt.day
df['purchase_month'] = ts.dt.month
df['is_weekend'] = (df['purchase_dayofweek'] >= 5).astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['purchase_hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['purchase_hour'] / 24.0)
df['month_sin'] = np.sin(2 * np.pi * df['purchase_month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['purchase_month'] / 12.0)

df['approval_delay_hours'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 3600.0
df['approval_delay_hours'] = df['approval_delay_hours'].clip(lower=0)

df['prep_sla_days'] = (
    df['min_shipping_limit'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400.0
df['prep_sla_days'] = df['prep_sla_days'].clip(lower=0)

# Đặc trưng đơn hàng & vật lý sản phẩm
df['freight_ratio'] = df['total_freight'] / (df['total_price'] + 1e-5)
df['price_per_item'] = df['total_price'] / df['total_items'].clip(lower=1)
df['freight_per_kg'] = df['total_freight'] / ((df['total_weight_g'] / 1000.0) + 0.1)
df['volumetric_weight_g'] = df['total_volume_cm3'] / 6.0

df['primary_category'] = df['primary_category'].fillna('unknown')
df['primary_payment_type'] = df['primary_payment_type'].fillna('unknown')
df['customer_state'] = df['customer_state'].fillna('UNK')
df['primary_seller_state'] = df['primary_seller_state'].fillna('UNK')
print('Feature engineering completed!')

## 4. Chia tập theo thời gian (Time-based Split), Target Encoding & Lưu file

In [ ]:
df = df.sort_values('order_purchase_timestamp').reset_index(drop=True)
n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Train: {len(train_df):,} ({train_df['order_purchase_timestamp'].min().date()} -> {train_df['order_purchase_timestamp'].max().date()})")
print(f"Val:   {len(val_df):,} ({val_df['order_purchase_timestamp'].min().date()} -> {val_df['order_purchase_timestamp'].max().date()})")
print(f"Test:  {len(test_df):,} ({test_df['order_purchase_timestamp'].min().date()} -> {test_df['order_purchase_timestamp'].max().date()})")

# Leakage-free OOF target encoding (fit chỉ trên Train)
te_cols = []
for col in ['primary_seller_id', 'route', 'customer_state', 'primary_category']:
    te_name = add_oof_target_encoding(
        train_df, val_df, test_df, col=col, target_col='delivery_time_days', smoothing=15.0
    )
    te_cols.append(te_name)

num_features = [
    'haversine_km', 'cust_lat', 'cust_lng', 'sell_lat', 'sell_lng',
    'is_same_state', 'is_same_city', 'purchase_hour', 'purchase_dayofweek',
    'purchase_day', 'purchase_month', 'is_weekend', 'hour_sin', 'hour_cos',
    'month_sin', 'month_cos', 'approval_delay_hours', 'prep_sla_days',
    'total_items', 'total_price', 'total_freight', 'num_unique_sellers',
    'num_unique_products', 'total_weight_g', 'max_weight_g', 'total_volume_cm3',
    'volumetric_weight_g', 'avg_photos_qty', 'freight_ratio', 'price_per_item',
    'freight_per_kg', 'max_installments', 'total_payment_value', 'num_payment_methods',
] + te_cols

cat_features = ['customer_state', 'primary_seller_state', 'primary_payment_type', 'primary_category']

# Impute missing numeric bằng median của Train
for col in num_features:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med)
    val_df[col] = val_df[col].fillna(med)
    test_df[col] = test_df[col].fillna(med)

keep_cols = ['order_id', 'order_purchase_timestamp', 'delivery_time_days', 'olist_estimated_days'] + num_features + cat_features
train_df[keep_cols].to_csv(os.path.join(OUT_DIR, 'train.csv'), index=False)
val_df[keep_cols].to_csv(os.path.join(OUT_DIR, 'val.csv'), index=False)
test_df[keep_cols].to_csv(os.path.join(OUT_DIR, 'test.csv'), index=False)

metadata = {
    'target_col': 'delivery_time_days',
    'benchmark_col': 'olist_estimated_days',
    'num_features': num_features,
    'cat_features': cat_features,
    'split_sizes': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)},
}
with open(os.path.join(OUT_DIR, 'feature_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'Saved processed datasets & metadata to {OUT_DIR}/')